# Load Data Smart: Combo WITH baqhiz=1766, WITHOUT=2201

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from statsmodels.stats.inter_rater import fleiss_kappa

PROJECT_DIR = Path(r"C:\Users\nesha\VSCODE_ALL\PBAAnalysisSentiment")
LABELING_DIR = PROJECT_DIR / "labeling"
DATA_OUTPUT_DIR = PROJECT_DIR / "data" / "labeled"
DATA_OUTPUT_DIR.mkdir(exist_ok=True)

ALL_LABELERS = ["shafwa", "hans", "gerald", "clay", "baqhiz"]
CATEGORIES = ["positif", "netral", "negatif"]
LEN_WITHOUT_BAQHIZ = 2201
LEN_WITH_BAQHIZ = 1766

# Load semua labeler (store full data, potong per kombinasi nanti)
print("📖 Loading semua 5 labeler (full data)...")
raw = {}
for name in ALL_LABELERS:
    file_path = LABELING_DIR / f"labels_{name}.csv"
    if file_path.exists():
        df = pd.read_csv(file_path, dtype=str, encoding="utf-8")
        raw[name] = df
        print(f"✔️ {name}: {len(df)} baris")

print(f"\n💡 Strategy:")
print(f"   Combo WITH baqhiz      → pakai {LEN_WITH_BAQHIZ} baris")
print(f"   Combo WITHOUT baqhiz   → pakai {LEN_WITHOUT_BAQHIZ} baris")

📖 Loading semua 5 labeler (full data)...
✔️ shafwa: 2201 baris
✔️ hans: 2201 baris
✔️ gerald: 2201 baris
✔️ clay: 2201 baris
✔️ baqhiz: 1766 baris

💡 Strategy:
   Combo WITH baqhiz      → pakai 1766 baris
   Combo WITHOUT baqhiz   → pakai 2201 baris


In [2]:
# Hitung kappa dengan min_len per kombinasi
def interp(k):
    return "Poor" if k < .2 else "Fair" if k < .4 else "Moderate" if k < .6 else "Substantial" if k < .8 else "Almost Perfect"

def calc_kappa_smart(combo):
    # Determine min length based on whether baqhiz in combo
    has_baqhiz = "baqhiz" in combo
    min_len = LEN_WITH_BAQHIZ if has_baqhiz else LEN_WITHOUT_BAQHIZ
    
    # Extract & normalize labels up to min_len
    labels_sub = pd.DataFrame({
        n: raw[n]["label"].iloc[:min_len].reset_index(drop=True)
                           .fillna("").str.lower().str.strip()
        for n in combo
    })
    
    # Filter valid rows (all labelers punya label)
    valid = labels_sub.isin(CATEGORIES).all(axis=1)
    labels_valid = labels_sub[valid]
    
    if len(labels_valid) == 0:
        return None, 0
    
    # Hitung kappa
    counts = np.column_stack([(labels_valid == c).sum(axis=1) for c in CATEGORIES])
    k = fleiss_kappa(counts, method="fleiss")
    return k, len(labels_valid)

print("\n" + "="*80)
print("🔬 HITUNG KAPPA SEMUA KOMBINASI (SMART MIN_LEN)")
print("="*80)

rows = []
for r in [2, 3, 4, 5]:
    for combo in combinations(ALL_LABELERS, r):
        k, n_data = calc_kappa_smart(combo)
        if k is not None:
            has_baqhiz = "baqhiz" in combo
            rows.append({
                "no": len(rows) + 1,
                "labelers": ", ".join(combo),
                "n_labeler": r,
                "min_len": LEN_WITH_BAQHIZ if has_baqhiz else LEN_WITHOUT_BAQHIZ,
                "n_valid": n_data,
                "kappa": round(k, 4),
                "interpretasi": interp(k),
            })

kappa_df = pd.DataFrame(rows).sort_values("kappa", ascending=False).reset_index(drop=True)
kappa_df["no"] = kappa_df.index + 1
print(kappa_df[["no", "labelers", "min_len", "n_valid", "kappa", "interpretasi"]].to_string())


🔬 HITUNG KAPPA SEMUA KOMBINASI (SMART MIN_LEN)
    no                            labelers  min_len  n_valid   kappa interpretasi
0    1                        shafwa, hans     2201     2201  0.7449  Substantial
1    2                shafwa, hans, gerald     2201     2200  0.7047  Substantial
2    3                        hans, gerald     2201     2200  0.6965  Substantial
3    4                      shafwa, gerald     2201     2200  0.6746  Substantial
4    5          shafwa, hans, gerald, clay     2201     2200  0.5669     Moderate
5    6                shafwa, gerald, clay     2201     2200  0.5293     Moderate
6    7                  shafwa, hans, clay     2201     2201  0.5283     Moderate
7    8                  hans, gerald, clay     2201     2200  0.5187     Moderate
8    9                        gerald, clay     2201     2200  0.4669     Moderate
9   10                        shafwa, clay     2201     2201  0.4595     Moderate
10  11                          hans, clay     220

In [3]:
# Pilih kombinasi
print("\n" + "="*80)
print("🎯 PILIH KOMBINASI LABELER")
print("="*80)

default = 1
best = kappa_df.loc[0]
print(f"\nTerbaik: no {default} - {best['labelers']} (kappa={best['kappa']}, data={best['n_valid']})")

ans = input(f"Pilih nomor [Enter = {default}]: ").strip()
no = int(ans) if ans else default
row = kappa_df.loc[no-1]
chosen = [x.strip() for x in row["labelers"].split(",")]
min_len = row["min_len"]

print(f"\n✅ Dipilih: {chosen}")
print(f"   Min length: {min_len}")
print(f"   Kappa: {row['kappa']} ({row['interpretasi']})")
print(f"   Valid data: {row['n_valid']}")

# Extract labels untuk kombinasi yang dipilih
sub = pd.DataFrame({
    n: raw[n]["label"].iloc[:min_len].reset_index(drop=True)
                       .fillna("").str.lower().str.strip()
    for n in chosen
})

valid = sub.isin(CATEGORIES).all(axis=1)
print(f"   Valid rows: {valid.sum()} dari {min_len}")

votes = pd.DataFrame({c: (sub == c).sum(axis=1) for c in CATEGORIES})
n = len(chosen)
top = votes.max(axis=1)
n_top = (votes == top.values[:, None]).sum(axis=1)
modus = votes.idxmax(axis=1)


🎯 PILIH KOMBINASI LABELER

Terbaik: no 1 - shafwa, hans (kappa=0.7449, data=2201)

✅ Dipilih: ['shafwa', 'hans']
   Min length: 2201
   Kappa: 0.7449 (Substantial)
   Valid data: 2201
   Valid rows: 2201 dari 2201


In [4]:
# Preview strategi
print("\n" + "="*80)
print("📊 PREVIEW STRATEGI AGREGASI")
print("="*80)

strategies = {
    "1": ("unanimous", valid & (top == n), modus, "unanimous"),
    "2": (f"majority vote", valid & (top > n / 2), modus, "majority"),
    "3": ("plurality", valid & (n_top == 1), modus, "plurality"),
}

if n == 2:
    a, b = chosen
    agree = valid & (sub[a] == sub[b])
    mixed = sub[a].where(agree, "DISAGREEMENT")
    strategies = {
        "1": ("agreement only", agree, modus, "agreement"),
        "2": ("all + DISAGREEMENT marked", valid, mixed, "all_marked"),
        "3": (f"all + label from {a}", valid, sub[a], f"all_{a}"),
        "4": (f"all + label from {b}", valid, sub[b], f"all_{b}"),
    }

preview = []
for key, (desc, mask, fin, tag) in strategies.items():
    d = fin[mask].value_counts()
    preview.append({
        "opsi": key,
        "strategi": desc,
        "positif": int(d.get("positif", 0)),
        "netral": int(d.get("netral", 0)),
        "negatif": int(d.get("negatif", 0)),
        "disagreement": int(d.get("DISAGREEMENT", 0)),
        "total": int(mask.sum()),
    })

preview_df = pd.DataFrame(preview).set_index("opsi")
print("\n" + preview_df.to_string())


📊 PREVIEW STRATEGI AGREGASI

                       strategi  positif  netral  negatif  disagreement  total
opsi                                                                          
1                agreement only      390      84     1485             0   1959
2     all + DISAGREEMENT marked      390      84     1485           242   2201
3       all + label from shafwa      455     199     1547             0   2201
4         all + label from hans      441     126     1634             0   2201


In [5]:
# Pilih strategi & export
print("\n" + "="*80)
print("💾 PILIH STRATEGI & EXPORT")
print("="*80)

default = "1"
key = input(f"\nPilih opsi {list(strategies.keys())} [Enter = {default}]: ").strip() or default
desc, mask, fin, tag = strategies[key]

# Metadata dari shafwa (first min_len rows)
meta_cols = ["post_url", "username", "comment_text", "timestamp", "likes_count"]
meta = raw["shafwa"][meta_cols].iloc[:min_len].reset_index(drop=True)[mask].copy()

out = meta.copy()
for name in chosen:
    out[f"label_{name}"] = sub.loc[mask, name].values

for c in CATEGORIES:
    out[f"votes_{c}"] = votes.loc[mask, c].values

out["final_label"] = fin[mask].values

fname = f"labels_{'_'.join(chosen)}_{tag}.csv"
out_path = DATA_OUTPUT_DIR / fname
out.to_csv(out_path, index=False, encoding="utf-8")

print(f"\n✅ DONE!")
print(f"File: {out_path.name}")
print(f"Data: {len(out)} baris (UTF-8 encoded, emoji safe)")
print(f"\nDistribusi:")
for c in ["positif", "netral", "negatif"]:
    v = (out["final_label"] == c).sum()
    print(f"  {c:10s}: {v:5d} ({v/len(out)*100:5.1f}%)")
if "DISAGREEMENT" in out["final_label"].values:
    v = (out["final_label"] == "DISAGREEMENT").sum()
    print(f"  {'disagreement':10s}: {v:5d} ({v/len(out)*100:5.1f}%)")


💾 PILIH STRATEGI & EXPORT

✅ DONE!
File: labels_shafwa_hans_agreement.csv
Data: 1959 baris (UTF-8 encoded, emoji safe)

Distribusi:
  positif   :   390 ( 19.9%)
  netral    :    84 (  4.3%)
  negatif   :  1485 ( 75.8%)
